# ATML PA2 — Kaggle runner

Runs every required experiment with the repository's Python scripts (the notebook only orchestrates).

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **ON**. Use *Save Version → Save & Run All (commit)* so the run survives a closed browser tab.

**How it runs**
* Two lanes run in parallel, one per T4: **Lane A** (GPU 0) = Task 1 → Task 5 → Task 3, **Lane B** (GPU 1) = Task 2. Task 4 needs the standard DPO/PPO/GRPO adapters, so it starts when both lanes are done (generation and judging are split over the two GPUs again). With one GPU the lanes run one after another.
* Every step writes a marker in `results/.done/`; a re-run skips finished steps.
* After each task `/kaggle/working/taskN_results.zip` is written (results + figures + adapters); at the end `ATML_PA2_ALL_RESULTS.zip`.
* Live logs: `/kaggle/working/logs/<lane>.log` (the wait cell prints the last lines every 5 minutes).

**Resuming in a new session:** attach the previous version's output (or the zips uploaded as a dataset) as an *Input*. Cell 2 unpacks every `*results*.zip` under `/kaggle/input` into the repo, so finished steps are skipped.

**Expected wall-clock on T4 x2:** roughly 5–6 h for a fresh run (Lane A ≈ 4.5 h, Lane B ≈ 3 h, Task 4 ≈ 1 h).

In [ ]:
# ---- Settings ---------------------------------------------------------------
REPO_URL = "https://github.com/moizejaz-31/ATML-PA2.git"
BRANCH = "main"
WORK = "/kaggle/working/ATML-PA2"
FRESH_START = True       # delete results/figures that came with the git checkout (stale) unless restored from zips
PARALLEL_LANES = True    # use both T4s; set False to run everything on one GPU

In [ ]:
# 1. GPU check, clone / update code, install pinned packages, download + validate course assets
import os, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.exists(WORK):
    !git clone --branch {BRANCH} {REPO_URL} {WORK}
else:
    !cd {WORK} && git pull origin {BRANCH}
%cd {WORK}
!git log --oneline -3
!pip install -q -r requirements.txt
!python -m scripts.download_assets
!python -m scripts.validate_assets

In [ ]:
# 2. Restore previous progress (zips attached as Kaggle inputs) or clear stale results from the checkout
import glob, shutil, zipfile
zips = sorted(glob.glob("/kaggle/input/**/*results*.zip", recursive=True))
if zips:
    for z in zips:
        print("restoring", z)
        zipfile.ZipFile(z).extractall(WORK)
elif FRESH_START:
    for d in ["results/task1_dpo", "results/task2_ppo", "results/task3_grpo", "results/task4_safety",
              "results/task5_feedback", "results/.done"]:
        shutil.rmtree(d, ignore_errors=True)
    for f in glob.glob("report/figures/*.png") + glob.glob("report/tables/*"):
        os.remove(f)
    print("cleared stale results/figures from the checkout")
os.makedirs("results/.done", exist_ok=True)
os.makedirs("/kaggle/working/logs", exist_ok=True)
print("done markers:", sorted(os.listdir("results/.done")))

In [ ]:
# 3. Write the lane scripts (each step: skip if its marker exists; a failing step stops its lane)
COMMON = r"""#!/bin/bash
cd /kaggle/working/ATML-PA2
export PYTHONUNBUFFERED=1 TOKENIZERS_PARALLELISM=false
step () {  # step <name> <command...>
  local name=$1; shift
  if [ -f results/.done/$name ]; then echo "[skip] $name"; return 0; fi
  echo "========== [$(date +%H:%M:%S)] START $name"
  if "$@"; then touch results/.done/$name; echo "========== [$(date +%H:%M:%S)] DONE $name";
  else echo "========== FAILED $name"; exit 1; fi
}
pack () { zip -qr /kaggle/working/$1 results report outputs -x "*.ipynb_checkpoints*"; echo "[zip] /kaggle/working/$1"; }
"""
LANE_A = COMMON + r"""
# ---------------- Task 1: DPO ----------------
step t1_preprocess   python -m task1_dpo.preprocess --config configs/dpo.yaml
step t1_eval_sft     python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter none --name sft_reference
step t1_train_std    python -m task1_dpo.train --config configs/dpo.yaml --run-name standard
step t1_eval_std     python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard
step t1_beta         python -m task1_dpo.ablate_beta --config configs/dpo.yaml --skip-train
step t1_length       python -m task1_dpo.analyze_length --config configs/dpo.yaml --skip-train
pack task1_results.zip
# ---------------- Task 5: RLVR vs RLAIF ----------------
step t5_gsm          python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset gsm
step t5_diag         python -m task5_feedback.score_perturbations --config configs/feedback.yaml
step t5_transfer     python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset transfer
step t5_compare      python -m task5_feedback.compare_feedback --config configs/feedback.yaml
pack task5_results.zip
# ---------------- Task 3: GRPO ----------------
step t3_eval_mid     python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter checkpoints/grpo_midpoint_policy --name midpoint
step t3_train_std    python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard
step t3_eval_std     python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard
step t3_group_size   python -m task3_grpo.analyze_group_size --config configs/grpo.yaml
step t3_norm         python -m task3_grpo.compare_normalization --config configs/grpo.yaml --skip-train
pack task3_results.zip
"""
LANE_B = COMMON + r"""
# ---------------- Task 2: PPO ----------------
step t2_eval_mid     python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter checkpoints/ppo_midpoint_policy --name midpoint
step t2_eval_sft     python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter none --name sft_reference
step t2_train_std    python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard
step t2_eval_std     python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard
step t2_clipping     python -m task2_ppo.analyze_clipping --config configs/ppo.yaml
step t2_kl           python -m task2_ppo.ablate_kl --config configs/ppo.yaml
pack task2_results.zip
"""
T4_GPU0 = COMMON + r"""
step t4_gen_sft_dpo   python -m task4_safety.generate_responses --config configs/feedback.yaml --policies sft dpo --skip-existing
step t4_judge_sft_dpo python -m task4_safety.judge_responses --config configs/feedback.yaml --policies sft dpo --skip-existing
"""
T4_GPU1 = COMMON + r"""
step t4_gen_ppo_grpo   python -m task4_safety.generate_responses --config configs/feedback.yaml --policies ppo grpo --skip-existing
step t4_judge_ppo_grpo python -m task4_safety.judge_responses --config configs/feedback.yaml --policies ppo grpo --skip-existing
"""
for name, body in [("lane_a", LANE_A), ("lane_b", LANE_B), ("t4_gpu0", T4_GPU0), ("t4_gpu1", T4_GPU1)]:
    with open(f"/kaggle/working/{name}.sh", "w") as fh:
        fh.write(body)
    os.chmod(f"/kaggle/working/{name}.sh", 0o755)
print("lane scripts written")

In [ ]:
# 4. Launch helpers: one script per GPU, wait, print progress every 5 minutes
import time, torch
N_GPU = torch.cuda.device_count()

def launch(script, gpu):
    log = open(f"/kaggle/working/logs/{script}.log", "a")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(["bash", f"/kaggle/working/{script}.sh"], stdout=log, stderr=subprocess.STDOUT, env=env)

def tail(script, n=3):
    try:
        lines = open(f"/kaggle/working/logs/{script}.log", errors="replace").read().splitlines()
        return "\n".join("    " + l[-200:] for l in lines[-n:])
    except FileNotFoundError:
        return "    (no log yet)"

def run_parallel(scripts, every=300):
    procs = {}
    if PARALLEL_LANES and N_GPU >= 2:
        procs = {s: launch(s, i) for i, s in enumerate(scripts)}
    else:
        for s in scripts:                      # sequential fallback on one GPU
            procs[s] = launch(s, 0)
            while procs[s].poll() is None:
                time.sleep(every)
                print(f"--- {time.strftime('%H:%M:%S')} [{s}]\n{tail(s)}", flush=True)
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every)
        print(f"--- {time.strftime('%H:%M:%S')} ---", flush=True)
        for s in procs:
            print(f"[{s}] running={procs[s].poll() is None}\n{tail(s)}", flush=True)
    for s, p in procs.items():
        print(f"[{s}] exit code {p.returncode}  (log: /kaggle/working/logs/{s}.log)")
        if p.returncode != 0:
            print(tail(s, 40))
    return {s: p.returncode for s, p in procs.items()}

print("GPUs:", N_GPU)

## Tasks 1, 2, 3 and 5 (two lanes)

In [ ]:
codes_ab = run_parallel(["lane_a", "lane_b"])

## Task 4 — safety calibration (needs the standard DPO / PPO / GRPO adapters)

In [ ]:
missing = [s for s in ["t1_train_std", "t2_train_std", "t3_train_std"] if not os.path.exists(f"results/.done/{s}")]
assert not missing, f"standard adapters missing for {missing} - check the lane logs above"
codes_t4 = run_parallel(["t4_gpu0", "t4_gpu1"])
!python -m task4_safety.make_audit_sheet --config configs/feedback.yaml
!python -m task4_safety.evaluate_safety --config configs/feedback.yaml
!zip -qr /kaggle/working/task4_results.zip results report outputs

## Report tables, cross-task figures, qualitative candidates, final archive

In [ ]:
!python -m common.visualize_all
!python -m scripts.extract_qualitative
!zip -qr /kaggle/working/ATML_PA2_ALL_RESULTS.zip results report outputs -x "*.ipynb_checkpoints*"
!ls -la /kaggle/working/*.zip
print("Done. Download ATML_PA2_ALL_RESULTS.zip from the Output tab.")

### After the run
1. Unzip `ATML_PA2_ALL_RESULTS.zip` into your local repo (it overwrites `results/`, `report/` and `outputs/`).
2. Fill `results/task4_safety/manual_audit_sheet.csv` (column `manual_label`, one of `SAFE_ANSWER, JUSTIFIED_REFUSAL, UNSAFE_COMPLIANCE, OVER_REFUSAL, AMBIGUOUS`) **without** opening `manual_audit_key.csv`, then run locally (CPU is enough):
   `python -m task4_safety.evaluate_safety --config configs/feedback.yaml`, `python -m common.visualize_all`, `python -m scripts.extract_qualitative`.
3. Commit code + `results/` + `report/` (adapters in `outputs/` stay git-ignored).